# 01 - Data audit (Hindustani)

CPU runtime is enough. Run top to bottom. Downloads two Zenodo zips (HMD 3.6 GB, Saraga 4.1 GB), keeps only the
pitch/tonic/label files and deletes the zips. Everything small (catalog, MusicBrainz cache, split file) is saved to
persistent storage.

What this answers: how many recordings / hours / artists per raga, which are vocal, which raga names
are near-duplicates, and (last cell) writes the recording-level train/val/test split.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/vishwesh-pujari/raaga-detector.git"
REPO_DIR = "/content/raaga-detector" if os.path.exists("/content") else "/kaggle/working/raaga-detector"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs are not always visible to a running kernel

# Google Drive is OPTIONAL. Mounting it only buys persistence of a few small files (a results
# table, a split file, some caches -- well under 100 MB) between sessions. Skip it if you are
# short on Drive space: everything below falls back to Colab's own local disk, and the one cell
# at the end of notebook 01 downloads the two files you actually need to your computer directly.
MOUNT_DRIVE = False  # flip to True if you have Drive space and want caches to persist
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount failed or was skipped, continuing on local disk:", e)

In [ ]:
from pathlib import Path
import pandas as pd
from raaga import env
from raaga.data import catalog, hmd, saraga, splits, musicbrainz

RAW, ROOT = env.raw_data_home(), env.root()
print("persistent:", ROOT, "| raw:", RAW)
print(f"{'Drive is mounted -- these are saved to Drive and persist between sessions.' if 'drive' in str(ROOT) else 'Drive is not mounted -- these are on local disk and vanish when the runtime resets.'}")
print("Raw downloads (a few GB, deleted down to the pitch/tonic files) always stay on local disk, never on Drive.")

## 1. Download
HMD = CompMusic Hindustani Music Dataset (300 recordings, 30 ragas x 10). Only the open pitch/tonic files are kept; the audio is restricted and not needed for the pitch baseline. (mirdata is not used: its HMD index lists only the Carnatic half.)

In [ ]:
hmd.download(RAW)  # 3.6 GB zip -> keeps ~2.3 GB of Hindustani pitch/tonic. Resumable; run again if interrupted

Saraga (108 tracks, 61 ragas, CC BY-NC-SA 4.0, non-commercial) is optional. Its zip is 4.1 GB (mostly mp3, which we discard). Set `False` to skip.

In [ ]:
INCLUDE_SARAGA = True
if INCLUDE_SARAGA:
    saraga.download(RAW)

## 2. Catalog

In [ ]:
CFG = Path(REPO_DIR) / "configs"
df = catalog.build_catalog(RAW, include_saraga=INCLUDE_SARAGA, raga_aliases=CFG / "raga_aliases.yaml", artist_aliases=CFG / "artist_aliases.yaml")
print(len(df), "recordings,", df.raga.nunique(), "ragas,", int(df.has_pitch.sum()), "with pitch")
display(df.groupby("dataset").size().rename("recordings"))
display(df.head(3).T)

## 3. Vocal or not?
Saraga's metadata names the lead instrument, so it is already classified. HMD has no instrument metadata (its 55 artists look like vocalists, except possibly Gopal Mishra), so look each MBID up on MusicBrainz (1 request/s, ~6 min for 300, cached). Coverage is patchy: `unknown` means MusicBrainz lists no vocal/instrument credit.

In [ ]:
from tqdm.auto import tqdm
m = df.dataset == "hmd"
res = {}
for mbid in tqdm(df.loc[m, "mbid"].dropna().unique()):
    try:
        res[mbid] = musicbrainz.fetch(mbid, ROOT / "musicbrainz")
    except Exception as e:
        print("failed", mbid, e)
df.loc[m, "vocal_class"] = df.loc[m, "mbid"].map(lambda x: res.get(x, {}).get("vocal_class", "unknown"))
df.loc[m, "instruments"] = df.loc[m, "mbid"].map(lambda x: ", ".join(res.get(x, {}).get("instruments", [])))
print(df.vocal_class.value_counts())
display(df[df.vocal_class == "instrumental"][["uid", "dataset", "raga", "artist", "instruments"]])
print(df.groupby("dataset").vocal_class.value_counts())

## 4. Per-raga table

In [ ]:
df = catalog.add_pitch_stats(df)  # reads every pitch file, ~1-2 min
df["hours"] = df.duration_s / 3600
df["voiced_frac"] = df.voiced_s / df.duration_s
df.to_csv(ROOT / "catalog.csv", index=False)
table = df.groupby("raga").agg(
    recordings=("uid", "size"), artists=("artist", "nunique"), concerts=("concert", "nunique"),
    hours=("hours", "sum"), voiced_frac=("voiced_frac", "mean"),
    vocal=("vocal_class", lambda s: (s == "vocal").sum()), unknown=("vocal_class", lambda s: (s == "unknown").sum()),
).sort_values("recordings", ascending=False)
pd.set_option("display.max_rows", 200)
display(table.round(2))
print("total hours:", round(df.hours.sum(), 1), "| tonic missing:", int(df.tonic_hz.isna().sum()))

In [ ]:
import difflib, itertools
names = sorted(df.raga.dropna().unique())
pairs = [(a, b) for a, b in itertools.combinations(names, 2) if difflib.SequenceMatcher(None, a, b).ratio() > 0.8]
print("near-duplicate raga names (add to configs/raga_aliases.yaml if the same raga):", pairs or "none")

## 5. Split (recording-level, grouped by concert)
Fold 4 is the held-out test set: do not evaluate on it until the final numbers. Folds 0-3 are for cross-validation.
`vocal_ok` includes `unknown` for now (MusicBrainz coverage is patchy); tighten it after looking at section 3.

In [ ]:
MIN_RECORDINGS = 8
VOCAL_OK = ("vocal", "unknown")
use = catalog.usable(df, vocal_ok=VOCAL_OK)
print("usable:", len(use), "of", len(df))
use = splits.filter_ragas(use, MIN_RECORDINGS)
print("after min", MIN_RECORDINGS, "recordings/raga:", len(use), "recordings,", use.raga.nunique(), "ragas")
folded = splits.assign_folds(use, seed=0, group_col="concert")
print("problems:", splits.validate_folds(folded) or "none")
print("ragas from fewer concerts than folds (cannot be in every fold):")
display(splits.thin_ragas(folded))
print("share of test recordings whose artist is also in train:", round(splits.artist_overlap(folded), 2))
display(pd.crosstab(folded.raga, folded.fold))
(ROOT / "splits").mkdir(exist_ok=True)
splits.save_split(folded, ROOT / "splits" / "v1.csv")
print("saved", ROOT / "splits" / "v1.csv")

The cell below downloads the two small files you need (`catalog.csv`, `splits/v1.csv`) straight to your computer, so nothing has to live on Drive. Put `splits/v1.csv` in the repo at `data/splits/v1.csv` and commit it. Also paste the tables printed above back into the chat: the plan gets updated from them.

In [ ]:
try:
    from google.colab import files
    files.download(str(ROOT / "catalog.csv"))
    files.download(str(ROOT / "splits" / "v1.csv"))
except ImportError:
    print("Not on Colab -- files are already on local disk at:", ROOT / "catalog.csv", "and", ROOT / "splits" / "v1.csv")